# COPT-MT: Paired Neural-Guidance Pilot trên Google Colab (Milestone M1)

Notebook này hướng dẫn chạy thực nghiệm M1 trên Google Colab với GPU T4 miễn phí:
Đo lường lợi ích thực tế của neural guidance (checkpoint COPT-MIS) trong exact Maximum Clique search sau khi trừ chi phí feature extraction và inference.

> **Bước bắt buộc trước khi chạy:**
> Vào menu **Runtime -> Change runtime type** (Thời gian chạy -> Thay đổi loại thời gian chạy):
> - Phần cứng: Chọn **T4 GPU**
> - Bấm **Save** (Lưu).

In [ ]:
# 1. Kiểm tra môi trường GPU
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU Model:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Vui lòng bật GPU trong Runtime -> Change runtime type -> T4 GPU trước khi tiếp tục!")

In [ ]:
# 2. Clone mã nguồn nếu runtime chưa có repo; không xóa thư mục đang tồn tại
from pathlib import Path
import subprocess
repo = Path('/content/COPT-MT')
if not repo.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/ghugn/COPT.git', str(repo)])
else:
    print('Dùng repo hiện có:', repo)
%cd /content/COPT-MT
!git log -n 1 --oneline

In [ ]:
# 3. Cài đặt các thư viện phụ thuộc của COPT
%pip install -q \
    hydra-core \
    lightning \
    torchmetrics \
    torch-geometric \
    hydra-submitit-launcher rootutils networkx numba POT rich wandb \
    yacs einops dwave-networkx dimod ogb performer-pytorch python-sat

In [ ]:
# 4. Cài đặt PyG extensions (torch-scatter, torch-sparse) đúng bản Torch/CUDA của Colab
import subprocess
import sys
import torch

torch_version = torch.__version__.split("+")[0]
cuda_tag = "cu" + torch.version.cuda.replace(".", "")
wheel_index = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
print("Đang tải prebuilt wheels từ:", wheel_index)

cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "--no-index", f"--find-links={wheel_index}",
    "torch-scatter", "torch-sparse"
]
subprocess.check_call(cmd)
print("Cài đặt torch-scatter và torch-sparse thành công!")

In [ ]:
# 5. Tải file bundle (chỉ ~6.5 MB) lên Colab và giải nén
# Bạn có thể chọn file guidance_kaggle_bundle.zip từ máy tính hoặc kéo thả vào thanh Files bên trái
from pathlib import Path
import zipfile
import os
from google.colab import files

%cd /content
bundle_path = Path("/content/guidance_kaggle_bundle.zip")

if not bundle_path.exists():
    print("Chưa thấy file zip. Vui lòng bấm nút 'Choose Files' dưới đây để tải lên guidance_kaggle_bundle.zip:")
    uploaded = files.upload()

assert bundle_path.exists(), "Không tìm thấy file guidance_kaggle_bundle.zip tại /content!"

%cd /content/COPT-MT
with zipfile.ZipFile(bundle_path) as z:
    z.extractall("/content/COPT-MT")
print("Đã giải nén bundle thành công vào /content/COPT-MT!")

CHECKPOINT = "/content/COPT-MT/artifacts/checkpoints/mis_rb_small/epoch_189.ckpt"
assert Path(CHECKPOINT).exists(), f"Không tìm thấy checkpoint tại {CHECKPOINT}"
print("Đường dẫn checkpoint sẵn sàng:", CHECKPOINT)

In [ ]:
# 6. Kiểm tra tính đúng đắn toán học của Solver (Unit Tests)
%cd /content/COPT-MT
!python -m unittest experiments.guidance.test_solver -v

In [ ]:
# 7. Bước 1: Smoke test COPT trên GPU Colab (xác thực model inference)
%cd /content/COPT-MT
!rm -rf /content/guidance_smoke_copt
!python -u -m experiments.guidance.pilot \
    --checkpoint "{CHECKPOINT}" \
    --device cuda \
    --sizes 30 40 \
    --densities 0.5 0.7 \
    --graphs-per-cell 1 \
    --states-per-graph 2 \
    --repeats 2 \
    --seconds 1 \
    --output /content/guidance_smoke_copt

from pathlib import Path
smoke_summary = Path("/content/guidance_smoke_copt/summary.json")
if smoke_summary.exists():
    print("Smoke test thành công! Summary:")
    print(smoke_summary.read_text())

In [ ]:
# 8. Bước 2: Chạy M1 và ghi trực tiếp vào Google Drive để không mất khi runtime ngắt
# Thiết lập: 60 đồ thị ER (sizes 40, 60, 80, 120; densities 0.3, 0.5, 0.7)
# So sánh cùng state, cùng incumbent: Structural vs Random vs COPT
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
drive.mount('/content/drive')
result_root = Path('/content/drive/MyDrive/COPT-MT-results')
result_root.mkdir(parents=True, exist_ok=True)
run_stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_utc')
RUN_OUTPUT = result_root / f'guidance_pilot_m1_{run_stamp}'
print('Kết quả sẽ được lưu tại:', RUN_OUTPUT)
%cd /content/COPT-MT
!python -u -m experiments.guidance.pilot \
    --checkpoint "{CHECKPOINT}" \
    --device cuda \
    --sizes 40 60 80 120 \
    --densities 0.3 0.5 0.7 \
    --graphs-per-cell 5 \
    --states-per-graph 4 \
    --repeats 3 \
    --seconds 2 \
    --node-limit 200000 \
    --output "{RUN_OUTPUT}"

In [ ]:
# 9. Tải kết quả về máy tính cá nhân
import shutil
from pathlib import Path
from google.colab import files

output = Path(RUN_OUTPUT)
if output.exists():
    summary_file = output / "summary.json"
    if summary_file.exists():
        print("=== KẾT QUẢ SUMMARY M1 ===")
        print(summary_file.read_text())
    
    archive_path = shutil.make_archive("/content/guidance_pilot_m1", "zip", output)
    print(f"Đã nén kết quả thành: {archive_path}")
    print("Đang tự động kích hoạt tải file về máy tính...")
    files.download(archive_path)
else:
    print("Không tìm thấy thư mục kết quả. Hãy kiểm tra lại log bước 8.")